In [1]:
#This setup installs required Python libraries and system dependencies for building a document processing pipeline with OCR, PDF handling, and LLM integration.
!pip install -q langchain langchain-groq python-dotenv pdfplumber pydantic pytesseract pillow pdf2image
!apt-get install -y tesseract-ocr
!apt-get install -y poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.


In [2]:
#Step 2: Setup Environment
#.env
import os #system operations
from google.colab import userdata #user secrets
from dotenv import load_dotenv #environment loading
import json #data serialization
import uuid #unique identifiers
import sqlite3 #database management
import traceback #error debugging

from datetime import datetime

load_dotenv()  # optional if you have .env
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')

In [3]:
#step3: Initialize LLM (Streaming Enabled)

from langchain_groq import ChatGroq #Used to connect to Groq-hosted LLMs through LangChain

llm = ChatGroq( #Model Instance
    model="openai/gpt-oss-120b", #large open-source-style model hosted on Groq
    temperature=0, #0 - It defines same Input and output
    streaming=True, #Enables real-time output. Tokens are returned piece by piece instead of all at once
    groq_api_key=os.getenv("GROQ_API_KEY") #Fetches your API key from environment variables
)

In [4]:
#step4: Define Structured JSON Schema with Pydantic - data validation and typing - hints
from pydantic import BaseModel, Field
from typing import Optional, List

class DocumentInfo(BaseModel): # Defines a model for document information
    document_id: Optional[str] = None
    document_type: str
    issuing_authority: Optional[str] = None
    recipient: Optional[str] = None
    subject: Optional[str] = None
    date: Optional[str] = None
    related_documents: Optional[List[str]] = []
    confidence: Optional[float] = None
    needs_review: Optional[bool] = False



In [5]:
# STEP 5: Document Loader (PDF + OCR + Metadata)
# pdfplumber → PDF extraction
# pytesseract → text recognition
# PIL.Image → image processing
# pdf2image.convert_from_path → PDF conversion

import pdfplumber, pytesseract, os, traceback
from PIL import Image
from pdf2image import convert_from_path
from datetime import datetime

class DocumentLoader:

    def load(self, file_path: str):
        try:
            ext = file_path.split(".")[-1].lower()

            if ext == "pdf":
                return self._load_pdf(file_path)
            elif ext in ["png", "jpg", "jpeg"]:
                return self._load_image(file_path)
            else:
                return {"text": "", "metadata": {}, "error": "Unsupported file"}

        except Exception as e:
            return {"text": "", "metadata": {}, "error": str(e)}

    def _load_pdf(self, path):
        text = ""
        metadata = {"file": os.path.basename(path), "type": "pdf"}

        try:
            with pdfplumber.open(path) as pdf:
                metadata["pages"] = len(pdf.pages)

                for page in pdf.pages:
                    t = page.extract_text()
                    if not t:
                        images = convert_from_path(path)
                        for img in images:
                            t += pytesseract.image_to_string(img)
                    text += t or ""

        except Exception as e:
            return {"text": "", "metadata": metadata, "error": str(e)}

        return {"text": text, "metadata": metadata, "error": None}

    def _load_image(self, path):
        try:
            img = Image.open(path)
            text = pytesseract.image_to_string(img)

            metadata = {
                "file": os.path.basename(path),
                "type": "image",
                "created": datetime.fromtimestamp(os.path.getctime(path)).isoformat()
            }

            return {"text": text, "metadata": metadata, "error": None}

        except Exception as e:
            return {"text": "", "metadata": {}, "error": str(e)}

In [6]:
#STEP 6: Classification (Router) - classifies documents into categories using an LLM and returns structured JSON output.
# json → data serialization
# SystemMessage → system instructions

import json
from langchain_core.messages import SystemMessage

def classify_cease(text):
    prompt = f"""
Classify document:

"Cease" → valid cease & desist
"Uncertain" → unclear
"Irrelevant" → not cease

Also give reason.

{text}

Return JSON:
{{"category":"", "confidence":0-1, "reason":""}}
"""
    res = llm.invoke([SystemMessage(content=prompt)])
    return json.loads(res.content)

In [7]:
#STEP 7: Extraction Tool -This function uses an LLM to convert unstructured text into structured JSON data.

def extract_info(text):
    prompt = f"""
Extract structured information from the document.

Return ONLY valid JSON.
Do NOT assign numeric values to text fields.
Do NOT return scores per field.

{text}

Required format:
{{
  "document_type": "string",
  "issuing_authority": "string",
  "recipient": "string",
  "subject": "string",
  "date": "YYYY-MM-DD",
  "confidence": 0.0
}}
"""
    res = llm.invoke([SystemMessage(content=prompt)])
    return res.content

In [8]:
#Add Normalization Function - This function cleans and standardizes extracted document data by fixing data types and ensuring a valid confidence value.
def normalize_data(data):
    # Ensure correct types
    fields = [
        "document_type",
        "issuing_authority",
        "recipient",
        "subject",
        "date"
    ]

    for f in fields:
        if isinstance(data.get(f), (int, float)):
            data[f] = str(data[f])  # convert wrong numeric → string

    # Fix confidence
    conf = data.get("confidence")

    if isinstance(conf, dict):
        # Take average if dict mistakenly returned
        values = [v for v in conf.values() if isinstance(v, (int, float))]
        data["confidence"] = sum(values) / len(values) if values else 0.5

    elif not isinstance(conf, (int, float)):
        data["confidence"] = 0.5  # fallback

    return data

In [9]:
#STEP 8: Validation Tool - validates extracted document data, detects errors, and adjusts confidence and review status accordingly.
def validate(data, error_log=None):
    errors = []
    confidence = data.get("confidence", 1.0)

    # -----------------------------
    # FIELD VALIDATION
    # -----------------------------
    if not data.get("document_type"):
        errors.append("Missing type")

    if data.get("date") and len(data["date"]) != 10:
        errors.append("Bad date format")

    # -----------------------------
    # ERROR LOG HANDLING
    # -----------------------------
    if error_log:
        for err in error_log:

            if err == "FILE_LOAD_ERROR":
                print("File load error → skipping file")

            elif err == "OCR_ERROR":
                print("OCR failed → request better image")

            elif err == "CLASSIFICATION_ERROR":
                print("Classification error → using fallback classifier")
                data["category"] = "Fallback"

            elif err == "EXTRACTION_ERROR":
                print("Extraction error → flagging for human review")
                data["needs_review"] = True

            errors.append(err)

    # -----------------------------
    # CONFIDENCE ADJUSTMENT
    # -----------------------------
    if errors:
        confidence -= 0.2 * len(errors)

    confidence = max(confidence, 0)

    return {
        "valid": len(errors) == 0,
        "confidence": confidence,
        "needs_review": len(errors) > 0 or confidence < 0.8
    }

In [10]:
#Step9: Add JSON Cleaner Function - Cleans and safely parses LLM-generated JSON by fixing formatting issues and handling errors.
import re
import json

def clean_json(raw):
    try:
        # Remove markdown ```json ``` blocks
        raw = re.sub(r"```json|```", "", raw).strip()

        # Fix incomplete JSON (very common)
        if not raw.endswith("}"):
            raw += "}"

        return json.loads(raw)

    except Exception as e:
        print("JSON Clean Failed:", e)
        print("RAW:", raw)
        return None



In [11]:
#Step10: ProcessingAgent - This class orchestrates the full document processing pipeline: extraction, cleaning, validation, and storage of structured document
#data using an LLM and Pydantic model.
class ProcessingAgent:
    def __init__(self):
        self.documents = {}

    def process(self, text):
        raw = extract_info(text)

        try:
            data = clean_json(raw)

            if not data:
                print("Invalid JSON, skipping")
                return None

            data = normalize_data(data)

            doc = DocumentInfo.model_validate(data)
            doc.document_id = str(uuid.uuid4())

            val = validate(doc.model_dump())
            doc.confidence = val["confidence"]
            doc.needs_review = val["needs_review"]

            self.documents[doc.document_id] = doc
            return doc

        except Exception as e:
            print("Processing Error:", e)
            print("RAW:", raw)
            return None

In [12]:
#Step11. SQLITE DATABASE - This class creates and manages a SQLite database to store, insert, and retrieve processed document data.

# Database Agent with path info
# ----------------------------
import sqlite3

class DatabaseAgent:
    def __init__(self, db_path="documents.db"):
        self.db_path = db_path
        self.conn = sqlite3.connect(db_path)
        self.cur = self.conn.cursor()
        self.cur.execute("""
        CREATE TABLE IF NOT EXISTS documents (
            id TEXT PRIMARY KEY,
            name TEXT,
            date_received TEXT,
            type TEXT,
            authority TEXT,
            recipient TEXT,
            subject TEXT,
            doc_date TEXT,
            confidence REAL,
            needs_review BOOLEAN,
            raw TEXT
        )
        """)
        self.conn.commit()

    def store(self, name, data):
        try:
            self.cur.execute("""
            INSERT INTO documents VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                data["document_id"],
                name,
                datetime.now().isoformat(),
                data.get("document_type"),
                data.get("issuing_authority"),
                data.get("recipient"),
                data.get("subject"),
                data.get("date"),
                data.get("confidence"),
                data.get("needs_review"),
                json.dumps(data)
            ))
            self.conn.commit()
            print(f"Stored in DB: {os.path.abspath(self.db_path)}")
        except sqlite3.IntegrityError:
            print("Duplicate document_id, skipping insert")

    def fetch_all(self):
        self.cur.execute("SELECT * FROM documents")
        return self.cur.fetchall()

In [13]:
# Step12. ARCHIVE + HITL - This code archives processed documents in Google Drive and sends uncertain cases for human review using separate agents.

import os
from datetime import datetime
from google.colab import drive

# ----------------------------
# Archive Agent with path info
# ----------------------------
class ArchiveAgent:
    def __init__(self, folder="archives"):
      drive.mount('/content/drive')
      archive_folder = "/content/drive/MyDrive/CapstoneProject/archives"
      os.makedirs(archive_folder, exist_ok=True)
      self.file_path = os.path.join(archive_folder, "archive.txt")

    def archive(self, name):
        with open(self.file_path, "a") as f:
            f.write(f"{datetime.now()} | {name}\n")
        print(f"Archived: {os.path.abspath(self.file_path)}")

class HumanAgent:
    def __init__(self):
        self.queue = []

    def review(self, name, text):
        self.queue.append({"name":name,"text":text})
        print("Sent to Human Review")

In [14]:
#Step13. AUDIT LOGGER - This class records, stores, and displays an audit trail of processed data with timestamps for tracking and debugging.
class Audit:
    def __init__(self):
        self.logs = []

    def log(self, data):
        data["time"] = datetime.now().isoformat()
        self.logs.append(data)

    def show(self):
        for l in self.logs:
            print(json.dumps(l, indent=2))


In [15]:
#Step14. Main Controller -This class acts as the main controller that runs the entire document processing workflow,
#including loading files, classifying, routing, storing, archiving, human review, auditing, and batch folder processing.
class MainController:
    def __init__(self):
        self.loader = DocumentLoader()
        self.processor = ProcessingAgent()
        self.db = DatabaseAgent()
        self.archive = ArchiveAgent()
        self.human = HumanAgent()
        self.audit = Audit()

    def run(self, file_path):
        doc = self.loader.load(file_path)

        if doc["error"]:
            print(f"Error: {file_path} → {doc['error']}")
            return

        text = doc["text"]
        name = doc["metadata"]["file"]

        # CLASSIFY
        cls = classify_cease(text)
        self.audit.log({"doc": name, "classification": cls})

        cat = cls["category"]

        # ROUTE
        if cat == "Cease":
            processed = self.processor.process(text)
            if processed:
                self.db.store(name, processed.dict())
            self.audit.log({"doc": name, "action": "stored"})

        elif cat == "Irrelevant":
            self.archive.archive(name)
            self.audit.log({"doc": name, "action": "archived"})

        elif cat == "Uncertain":
            self.human.review(name, text)
            self.audit.log({"doc": name, "action": "HITL"})

    #PROCESS ENTIRE FOLDER
    def run_folder(self, folder_path):
        print(f"\n Processing folder: {folder_path}\n")

        for root, _, files in os.walk(folder_path):
            for file in files:

                if not file.lower().endswith((".pdf", ".png", ".jpg", ".jpeg")):
                    continue

                full_path = os.path.join(root, file)

                print(f"\n Processing: {file}")

                try:
                    self.run(full_path)
                except Exception:
                    print(f"Failed: {file}")
                    traceback.print_exc()

        print("\n DATABASE:", os.path.abspath(self.db.db_path)) #DB AUDIT LOG
        print("\n ARCHIVE:", os.path.abspath(self.archive.file_path)) # ARCHIVE LOG
        print("\n HITL AUDIT LOGS:\n") # HITL AUDIT LOG

        hitl_logs = [
            log for log in self.audit.logs
            if log.get("action") == "HITL"
        ]

        if not hitl_logs:
            print("No HITL records found.")
        else:
            for log in hitl_logs:
                print(json.dumps(log, indent=2))


In [17]:

#Step15. RUN - File Specific
# MC = MainController()

# MC.run("/content/drive/MyDrive/Colab Notebooks/CapstoneProject/SampleFiles/bw_doc_4.pdf")
# print("\n AUDIT LOGS:")
# MC.audit.show()

# print("\n DB DATA:")
# print(MC.db.fetch_all())


#Step15. RUN (FOLDER)  - This code initializes the main controller, processes all documents in a folder, and then displays audit logs and stored database records.

MC = MainController()

folder_path = "/content/drive/MyDrive/Colab Notebooks/CapstoneProject/SampleFiles"

MC.run_folder(folder_path)

print("\n AUDIT LOGS:")
MC.audit.show()

print("\n DB DATA:")
print(MC.db.fetch_all())





Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

 Processing folder: /content/drive/MyDrive/Colab Notebooks/CapstoneProject/SampleFiles


 Processing: 01_copyright_infringement_photography.pdf


/tmp/ipykernel_13979/3696092620.py:32: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  self.db.store(name, processed.dict())


Stored in DB: /content/documents.db

 Processing: 04_defamation_online_review.pdf
Stored in DB: /content/documents.db

 Processing: 05_patent_infringement_medical_device.pdf
Stored in DB: /content/documents.db

 Processing: 02_trademark_infringement_tech.pdf
Stored in DB: /content/documents.db

 Processing: 03_trade_secret_misappropriation.pdf
Stored in DB: /content/documents.db

 Processing: 06_harassment_workplace.pdf
Stored in DB: /content/documents.db

 Processing: 07_software_license_violation.pdf
Stored in DB: /content/documents.db

 Processing: 08_non_compete_violation.pdf
Stored in DB: /content/documents.db

 Processing: 09_copyright_infringement_music.pdf
Stored in DB: /content/documents.db

 Processing: 10_breach_of_contract_nda.pdf
Stored in DB: /content/documents.db

 Processing: bw_doc_2.pdf
Archived: /content/drive/MyDrive/CapstoneProject/archives/archive.txt

 Processing: LOA5.pdf


/tmp/ipykernel_13979/3696092620.py:32: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  self.db.store(name, processed.dict())


Stored in DB: /content/documents.db

 Processing: bw_doc_5.pdf
Archived: /content/drive/MyDrive/CapstoneProject/archives/archive.txt

 Processing: LOA3.pdf


...

Stored in DB: /content/documents.db

 Processing: notice_3.pdf


/tmp/ipykernel_13979/3696092620.py:32: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  self.db.store(name, processed.dict())


Stored in DB: /content/documents.db

 Processing: notice_4.pdf


/tmp/ipykernel_13979/3696092620.py:32: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  self.db.store(name, processed.dict())


Stored in DB: /content/documents.db

 Processing: LOA9.pdf
Stored in DB: /content/documents.db

 DATABASE: /content/documents.db

 ARCHIVE: /content/drive/MyDrive/CapstoneProject/archives/archive.txt

 HITL AUDIT LOGS:

{
  "doc": "notice_5.pdf",
  "action": "HITL",
  "time": "2026-03-21T08:15:33.551644"
}

 AUDIT LOGS:
{
  "doc": "01_copyright_infringement_photography.pdf",
  "classification": {
    "category": "Cease",
    "confidence": 0.99,
    "reason": "The document is a formal cease\u2011and\u2011desist letter that explicitly demands the recipient stop all unauthorized use of copyrighted photographs, outlines specific actions to be taken, and warns of legal consequences for non\u2011compliance."
  },
  "time": "2026-03-21T08:11:19.919922"
}
{
  "doc": "01_copyright_infringement_photography.pdf",
  "action": "stored",
  "time": "2026-03-21T08:11:20.810104"
}
{
  "doc": "04_defamation_online_review.pdf",
  "classification": {
    "category": "Cease",
    "confidence": 0.99,
    "r

In [18]:
# STEP 15: Check Stored Data
rows = MC.db.fetch_all()

for r in rows:
    print(r)

# Archive contents
with open(MC.archive.file_path, "r") as f:
    archive_content = f.read().strip()
print("Archive Content:\n", archive_content if archive_content else "(empty)")

('ab78e845-c6d2-4ef9-96d4-bb9d1e34b026', '01_copyright_infringement_photography.pdf', '2026-03-21T08:11:20.800379', 'Cease and Desist Letter', 'Priya Nair Photography LLC', 'David Chen, Marketing Director, BrightLeaf Digital Agency', 'Copyright Infringement — Unauthorized Use of Photographs', '2026-03-10', 0.99, 0, '{"document_id": "ab78e845-c6d2-4ef9-96d4-bb9d1e34b026", "document_type": "Cease and Desist Letter", "issuing_authority": "Priya Nair Photography LLC", "recipient": "David Chen, Marketing Director, BrightLeaf Digital Agency", "subject": "Copyright Infringement \\u2014 Unauthorized Use of Photographs", "date": "2026-03-10", "related_documents": [], "confidence": 0.99, "needs_review": false}')
('fc80eb2c-c335-49b9-9d69-1c2285096a81', '04_defamation_online_review.pdf', '2026-03-21T08:11:22.149006', 'Cease and Desist Letter', 'Dr. Samantha Ellis', 'Mr. Brian Kessler', 'Defamation — False and Malicious Online Statements', '2026-03-03', 0.99, 0, '{"document_id": "fc80eb2c-c335-49b

In [19]:
#Check if DB file exists
!ls -lh documents.db
# !ls -l "/content/drive/MyDrive/CapstoneProject/archives"

-rw-r--r-- 1 root root 32K Mar 21 08:17 documents.db


In [20]:
#Convert DB rows into readable format:

rows = MC.db.fetch_all()

columns = [
    "id", "name", "date_received", "type", "authority",
    "recipient", "subject", "doc_date", "confidence",
    "needs_review", "raw"
]

for row in rows:
    record = dict(zip(columns, row))
    print(json.dumps(record, indent=2))
    print("="*50)

{
  "id": "ab78e845-c6d2-4ef9-96d4-bb9d1e34b026",
  "name": "01_copyright_infringement_photography.pdf",
  "date_received": "2026-03-21T08:11:20.800379",
  "type": "Cease and Desist Letter",
  "authority": "Priya Nair Photography LLC",
  "recipient": "David Chen, Marketing Director, BrightLeaf Digital Agency",
  "subject": "Copyright Infringement \u2014 Unauthorized Use of Photographs",
  "doc_date": "2026-03-10",
  "confidence": 0.99,
  "needs_review": 0,
  "raw": "{\"document_id\": \"ab78e845-c6d2-4ef9-96d4-bb9d1e34b026\", \"document_type\": \"Cease and Desist Letter\", \"issuing_authority\": \"Priya Nair Photography LLC\", \"recipient\": \"David Chen, Marketing Director, BrightLeaf Digital Agency\", \"subject\": \"Copyright Infringement \\u2014 Unauthorized Use of Photographs\", \"date\": \"2026-03-10\", \"related_documents\": [], \"confidence\": 0.99, \"needs_review\": false}"
}
{
  "id": "fc80eb2c-c335-49b9-9d69-1c2285096a81",
  "name": "04_defamation_online_review.pdf",
  "date_r